<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-07/AI_Modul_7.10_Praktikum_Gradient_Boosting.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 7.10: Praktikum Gradient Boosting

**Mata Kuliah:** AI Academy  
**Institusi:** Institut Pertanian Stiper (INSTIPER) Yogyakarta  
**Studi Kasus:** Peramalan Hasil Panen Tandan Buah Segar (TBS) Kelapa Sawit Skala Blok Perkebunan Menggunakan Gradient Boosting Regressor, Early Stopping, dan Analisis Residu

---

### Capaian Pembelajaran Praktikum
1. Mampu mengimplementasikan model Gradient Boosting Regressor dengan pustaka Scikit-Learn untuk peramalan target kontinu perkebunan.
2. Mampu menerapkan strategi regulasi *Shrinkage* (laju pembelajaran $\eta$) dan penghentian dini (*Early Stopping*) guna mencegah fenomena *overfitting*.
3. Mampu mengekstraksi dan menginterpretasikan peringkat kontribusi variabel agronomi (*Feature Importance Gain*).
4. Mampu melakukan komparasi kinerja prediktif head-to-head antara paradigma *Bagging* (Random Forest) versus *Boosting* (Gradient Boosting).

## 1. Import Pustaka & Konfigurasi Lingkungan


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Pengaturan visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("[OK] Seluruh pustaka Gradient Boosting berhasil dimuat!")


## 2. Pembangkitan Dataset Riwayat Panen TBS Kelapa Sawit

Data historis bulanan 1.200 blok kebun kelapa sawit produktif:
- `Curah_Hujan_Bln_Lalu`: Curah hujan 1 bulan sebelum panen (80.0 - 380.0 mm)
- `Defisit_Air_Kumulatif`: Akumulasi defisit air tanah 12 bulan terakhir (0.0 - 250.0 mm)
- `Dosis_Pupuk_Kalium`: Realisasi pupuk MOP / KCl (1.0 - 3.5 kg/pohon/tahun)
- `Umur_Tegakan_Thn`: Umur tanaman kelapa sawit (4.0 - 24.0 tahun)
- `Kerapatan_Pohon_Ha`: Kerapatan tegakan pohon per hektar / SPH (120 - 148 pohon)
- `Ketinggian_Blok`: Elevasi topografi blok kebun (25 - 320 mdpl)

Target:
- `Produksi_TBS_Ton_Ha`: Hasil panen riil bulanan per hektar (0.8 - 3.8 Ton/Ha)


In [ ]:
np.random.seed(42)
n_samples = 1200

hujan = np.random.uniform(80.0, 380.0, n_samples)
defisit_air = np.random.uniform(0.0, 250.0, n_samples)
pupuk_k = np.random.uniform(1.0, 3.5, n_samples)
umur = np.random.uniform(4.0, 24.0, n_samples)
sph = np.random.uniform(120.0, 148.0, n_samples)
elevasi = np.random.uniform(25.0, 320.0, n_samples)

# Pola biologis produksi TBS realistis (kurva non-linier puncak pada umur 9-14 tahun)
faktor_umur = np.exp(-0.5 * ((umur - 11.5) / 4.5)**2)
produksi_tbs = (
    0.85 +
    1.45 * faktor_umur +
    0.35 * (pupuk_k / 3.5) +
    0.003 * hujan -
    0.004 * defisit_air +
    0.008 * (sph - 120.0) -
    0.001 * (elevasi - 25.0) +
    np.random.normal(0, 0.12, n_samples)
)
produksi_tbs = np.clip(produksi_tbs, 0.8, 3.8)

fitur_names = ['Curah_Hujan_Bln_Lalu', 'Defisit_Air_Kumulatif', 'Dosis_Pupuk_Kalium', 
               'Umur_Tegakan_Thn', 'Kerapatan_Pohon_Ha', 'Ketinggian_Blok']

df_panen = pd.DataFrame({
    'Curah_Hujan_Bln_Lalu': hujan,
    'Defisit_Air_Kumulatif': defisit_air,
    'Dosis_Pupuk_Kalium': pupuk_k,
    'Umur_Tegakan_Thn': umur,
    'Kerapatan_Pohon_Ha': sph,
    'Ketinggian_Blok': elevasi,
    'Produksi_TBS_Ton_Ha': produksi_tbs
})

print(f"Dataset berhasil dibuat: {df_panen.shape[0]} baris sampel")
df_panen.head()


## 3. Eksplorasi Visual: Korelasi Non-Linier Umur & Iklim terhadap Panen


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Relasi Umur vs Produksi (Kurva Cembung Non-Linier)
axes[0].scatter(df_panen['Umur_Tegakan_Thn'], df_panen['Produksi_TBS_Ton_Ha'], color='#2e7d32', alpha=0.5, s=25)
axes[0].set_title('Hubungan Non-Linier: Umur Tanaman vs Produksi TBS', fontweight='bold')
axes[0].set_xlabel('Umur Tegakan Sawit (Tahun)')
axes[0].set_ylabel('Produksi TBS (Ton/Ha)')
axes[0].grid(True, linestyle=':', alpha=0.6)

# Relasi Defisit Air vs Produksi
axes[1].scatter(df_panen['Defisit_Air_Kumulatif'], df_panen['Produksi_TBS_Ton_Ha'], color='#d32f2f', alpha=0.5, s=25)
axes[1].set_title('Pengaruh Cekaman: Defisit Air Kumulatif vs Produksi TBS', fontweight='bold')
axes[1].set_xlabel('Defisit Air 12 Bulan Terakhir (mm)')
axes[1].set_ylabel('Produksi TBS (Ton/Ha)')
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## 4. Pembagian Data & Pelatihan Model Baseline (Regresi Linier)


In [ ]:
X = df_panen[fitur_names]
y = df_panen['Produksi_TBS_Ton_Ha']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Baseline: Regresi Linier Biasa
lr_baseline = LinearRegression()
lr_baseline.fit(X_train, y_train)

y_pred_lr = lr_baseline.predict(X_test)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Akurasi Regresi Linier (Baseline) - R²: {r2_lr:.4f} | RMSE: {rmse_lr:.4f} Ton/Ha")
print("Catatan: Regresi linier kesulitan menangkap kurva lengkung umur tanaman sawit.")


## 5. Pelatihan Gradient Boosting Regressor dengan Early Stopping

Konfigurasi model ansambel sekuensial teroptimasi:
- `n_estimators=200`: Jumlah maksimum pohon estimator
- `learning_rate=0.08`: Laju penyusutan (*shrinkage*)
- `max_depth=4`: Batasan kedalaman pohon dangkal
- `subsample=0.85`: Stochastic Gradient Boosting
- `n_iter_no_change=10`: Penghentian dini otomatis jika galat validasi stagnan


In [ ]:
gbr_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.08,
    max_depth=4,
    subsample=0.85,
    validation_fraction=0.15,
    n_iter_no_change=10,
    random_state=42
)

gbr_model.fit(X_train, y_train)

y_pred_gbr = gbr_model.predict(X_test)
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
r2_gbr = r2_score(y_test, y_pred_gbr)

print("=== EVALUASI MODEL GRADIENT BOOSTING REGRESSOR ===")
print(f"Jumlah Pohon Terpilih Aktual : {gbr_model.n_estimators_} (dihentikan otomatis via Early Stopping)")
print(f"Root Mean Squared Error (RMSE): {rmse_gbr:.4f} Ton/Ha")
print(f"Mean Absolute Error (MAE)     : {mae_gbr:.4f} Ton/Ha")
print(f"Koefisien Determinasi (R²)    : {r2_gbr:.4f} ({r2_gbr*100:.2f}% varians produksi terjelaskan)")


## 6. Analisis Peringkat Kepentingan Fitur (Feature Importance)


In [ ]:
importances = gbr_model.feature_importances_
indices = np.argsort(importances)[::-1]

df_importance = pd.DataFrame({
    'Variabel Agronomi': [fitur_names[i] for i in indices],
    'Kontribusi Gain (%)': importances[indices] * 100
})

print("=== PERINGKAT KONTRIBUSI VARIABEL TERHADAP TONASE TBS ===")
print(df_importance.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(x='Kontribusi Gain (%)', y='Variabel Agronomi', hue='Variabel Agronomi', data=df_importance, palette='Greens_r', legend=False, ax=ax)
ax.set_title('Peringkat Kepentingan Fitur Model Gradient Boosting Panen Sawit', fontweight='bold')
ax.set_xlabel('Derajat Kontribusi Relatif (%)')
plt.tight_layout()
plt.show()


## 7. Eksperimen Dinamika Konvergensi: Efek Laju Pembelajaran ($\eta$)


In [ ]:
rates = [0.02, 0.08, 0.40]
fig, ax = plt.subplots(figsize=(10, 5))

for lr in rates:
    m = GradientBoostingRegressor(n_estimators=120, learning_rate=lr, max_depth=4, random_state=42)
    m.fit(X_train, y_train)
    
    test_errors = [mean_squared_error(y_test, y_pred_stage) for y_pred_stage in m.staged_predict(X_test)]
    ax.plot(range(1, 121), test_errors, linewidth=2, label=f'Learning Rate = {lr}')

ax.set_title('Pengaruh Learning Rate (Shrinkage) terhadap Dinamika Test MSE', fontweight='bold')
ax.set_xlabel('Jumlah Iterasi Pohon (Estimators)')
ax.set_ylabel('Mean Squared Error (MSE)')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


## 8. Komparasi Head-to-Head: Random Forest vs Gradient Boosting


In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

df_komp = pd.DataFrame({
    'Model': ['Regresi Linier (Baseline)', 'Random Forest (Bagging)', 'Gradient Boosting (Boosting)'],
    'RMSE (Ton/Ha)': [rmse_lr, rmse_rf, rmse_gbr],
    'Skor R²': [r2_lr, r2_rf, r2_gbr],
    'Status Presisi': ['Kaku (Underfit)', 'Sangat Baik', 'State-of-the-Art Pemenang']
})

print("=== TABEL KOMPARASI PRESTASI MODEL PREDIKSI PANEN ===")
print(df_komp.to_string(index=False))

# Plot Aktual vs Prediksi
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred_gbr, color='#1b5e20', alpha=0.5, edgecolors='k', label='Gradient Boosting')
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Prediksi Sempurna (1:1)')
ax.set_title('Aktual vs Prediksi Hasil Panen TBS (Ton/Ha)', fontweight='bold')
ax.set_xlabel('Hasil Panen Aktual Lapangan (Ton/Ha)')
ax.set_ylabel('Prediksi Gradient Boosting (Ton/Ha)')
ax.legend()
plt.tight_layout()
plt.show()


## 9. Simulasi Peramalan Produksi 3 Blok Baru untuk Logistik Armada Truk


In [ ]:
blok_baru = pd.DataFrame([
    # Blok A: Umur prima (12 thn), pupuk tinggi, hujan cukup (Potensi Panen Puncak)
    {'Curah_Hujan_Bln_Lalu': 240.0, 'Defisit_Air_Kumulatif': 20.0, 'Dosis_Pupuk_Kalium': 3.2, 'Umur_Tegakan_Thn': 12.0, 'Kerapatan_Pohon_Ha': 142.0, 'Ketinggian_Blok': 45.0},
    # Blok B: Umur muda (5 thn), pupuk standar (Produksi Awal)
    {'Curah_Hujan_Bln_Lalu': 160.0, 'Defisit_Air_Kumulatif': 65.0, 'Dosis_Pupuk_Kalium': 2.0, 'Umur_Tegakan_Thn': 5.0, 'Kerapatan_Pohon_Ha': 140.0, 'Ketinggian_Blok': 80.0},
    # Blok C: Umur tua (22 thn), defisit air tinggi (Produksi Menurun)
    {'Curah_Hujan_Bln_Lalu': 95.0, 'Defisit_Air_Kumulatif': 180.0, 'Dosis_Pupuk_Kalium': 1.8, 'Umur_Tegakan_Thn': 22.0, 'Kerapatan_Pohon_Ha': 124.0, 'Ketinggian_Blok': 190.0}
])

pred_tonase = gbr_model.predict(blok_baru)
luas_blok_ha = 30.0 # Rata-rata 1 blok = 30 Hektar

print("=== PERENCANAAN LOGISTIK TRUK PENGANGKUTAN TBS PKS ===")
for i in range(len(blok_baru)):
    taksiran_total = pred_tonase[i] * luas_blok_ha
    jumlah_truk_8ton = int(np.ceil(taksiran_total / 8.0))
    print(f"\n--- Blok #{i+1} ---")
    print(f"Estimasi Produktivitas : {pred_tonase[i]:.2f} Ton/Ha")
    print(f"Estimasi Tonase Blok   : {taksiran_total:.2f} Ton TBS")
    print(f"Alokasi Armada Truk    : {jumlah_truk_8ton} Unit Truk Kapasitas 8-Ton")


## 10. Tugas Mandiri & Eksplorasi Mahasiswa

1. **Uji Parameter Max Depth**: Latih model Gradient Boosting dengan variasi `max_depth` antara 2, 4, 8, dan 15. Amati kurva galat latih vs uji. Mengapa pohon yang terlalu dalam justru menurunkan akurasi pada data uji?
2. **Eksplorasi Fungsi Rugi Alternatif**: Ubah parameter `loss='absolute_error'` (MAE) dan `loss='huber'`. Bandingkan ketahanannya ketika dataset disuntikkan pencilan ekstrem (*outliers*).
3. **Refleksi Besar Part 7 Menuju Part 8**: Anda telah menyelesaikan 10 modul Machine Learning pada data tabel terstruktur. Bagaimana jika kita dihadapkan pada rekaman video jutaan frame untuk mendeteksi brondolan sawit bergerak cepat di konveyor pabrik? Jelaskan mengapa kita membutuhkan Deep Learning & Convolutional Neural Networks (Part 8)!
